In [18]:
import sys
import json
import pickle
import argparse
import logging
import warnings
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# ML
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, mean_squared_error, r2_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    mean_absolute_error, log_loss
)
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.dummy import DummyClassifier, DummyRegressor

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

In [19]:
BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "bigdata" / "processed"
ARTIFACTS_DIR = BASE_DIR / "artifacts" / "experiments"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Подпапки артефактов
METRICS_DIR = ARTIFACTS_DIR / "metrics"
MODELS_DIR = ARTIFACTS_DIR / "models"
FIGURES_DIR = ARTIFACTS_DIR / "figures"
PREDICTIONS_DIR = ARTIFACTS_DIR / "predictions"
REPORTS_DIR = ARTIFACTS_DIR / "reports"
CONFIGS_DIR = ARTIFACTS_DIR / "configs"

for d in [METRICS_DIR, MODELS_DIR, FIGURES_DIR, PREDICTIONS_DIR, REPORTS_DIR, CONFIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [20]:
HORIZONS = [1, 5, 20, 200]
N_SPLITS = 3
TRIALS = 25
TEST_CUTOFF = "2023-01-01"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# GPU check
try:
    import torch
    TORCH_AVAILABLE = True
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"PyTorch доступен. Устройство: {DEVICE}")
except ImportError:
    TORCH_AVAILABLE = False
    DEVICE = None
    logger.warning("PyTorch не установлен. LSTM/Transformer недоступны.")

2026-05-19 03:13:10,404 [INFO] PyTorch доступен. Устройство: cuda


УТИЛИТЫ

In [21]:
def load_features():
    with open(PROCESSED_DIR / "feature_columns.txt") as f:
        return [line.strip() for line in f if line.strip()]

In [22]:
def temporal_split(df, feature_cols, target_col, cutoff_date=TEST_CUTOFF):
    """Корректный временной сплит по дате."""
    train_mask = df["date"] < cutoff_date
    test_mask = df["date"] >= cutoff_date
    X_train = df.loc[train_mask, feature_cols]
    X_test = df.loc[test_mask, feature_cols]
    y_train = df.loc[train_mask, target_col]
    y_test = df.loc[test_mask, target_col]
    valid_train = y_train.notna()
    valid_test = y_test.notna()
    return (X_train[valid_train], X_test[valid_test],
            y_train[valid_train], y_test[valid_test])

In [23]:
def prepare_data_for_model(X_train, X_test, model_name):
    """
    Подготовка данных под конкретную модель.
    CatBoost — NaN как есть.
    Остальные — median imputation (обученная на train, применённая к test).
    """
    if model_name == 'catboost':
        return X_train, X_test

    # Для остальных моделей — заполняем пропуски
    imputer = SimpleImputer(strategy='median')
    X_train_imp = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )
    X_test_imp = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_test.columns,
        index=X_test.index
    )
    return X_train_imp, X_test_imp

In [24]:
def get_gpu_params(model_name):
    try:
        import torch
        if torch.cuda.is_available():
            if model_name == 'catboost':
                return {'task_type': 'GPU', 'devices': '0'}
            elif model_name == 'lightgbm':
                return {'device': 'gpu'}
            elif model_name == 'xgboost':
                return {'device': 'gpu', 'gpu_platform_id': 0, 'gpu_device_id': 0}
    except ImportError:
        pass
    return {}

In [25]:
def save_json(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False, default=str)


def save_csv(df, path):
    df.to_csv(path, index=False)


def save_fig(name, fig=None):
    if fig is None:
        fig = plt.gcf()
    fig.tight_layout()
    path = FIGURES_DIR / name
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    logger.info(f"График сохранён: {path}")

МОДЕЛИ

In [26]:
MODEL_CONFIGS = {
    'catboost': {
        'binary': {
            'random_seed': RANDOM_SEED,
            'verbose': False,
            'allow_writing_files': False,
        },
        'return': {
            'random_seed': RANDOM_SEED,
            'verbose': False,
            'allow_writing_files': False,
        }
    },
    'lightgbm': {
        'binary': {
            'random_state': RANDOM_SEED,
            'verbose': -1,
        },
        'return': {
            'random_state': RANDOM_SEED,
            'verbose': -1,
        }
    },
    'xgboost': {
        'binary': {
            'random_state': RANDOM_SEED,
            'verbosity': 0,
        },
        'return': {
            'random_state': RANDOM_SEED,
            'verbosity': 0,
        }
    },
    'random_forest': {
        'binary': {
            'n_estimators':300,
            'max_depth': 10,
            'random_state': RANDOM_SEED,
            'n_jobs': -1,
        }
    },
    'logistic': {
        'binary': {
            'random_state': RANDOM_SEED,
            'n_jobs': -1,
        }
    },
    'ridge': {
        'return': {
            'random_state': RANDOM_SEED,
        }
    }
}

In [27]:
def get_model(model_name, target_type):
    """Фабрика моделей с фиксированными конфигами."""
    config = MODEL_CONFIGS.get(model_name, {}).get(target_type)
    if config is None:
        raise ValueError(f"Нет конфига для {model_name}/{target_type}")

    gpu_params = get_gpu_params(model_name)
    config = {**config, **gpu_params}

    if target_type == 'binary':
        if model_name == 'catboost':
            return CatBoostClassifier(**config)
        elif model_name == 'lightgbm':
            return LGBMClassifier(**config)
        elif model_name == 'xgboost':
            return XGBClassifier(**config)
        elif model_name == 'random_forest':
            return RandomForestClassifier(**config)
        elif model_name == 'logistic':
            return LogisticRegression(**config)
    else:
        if model_name == 'catboost':
            return CatBoostRegressor(**config)
        elif model_name == 'lightgbm':
            return LGBMRegressor(**config)
        elif model_name == 'xgboost':
            return XGBRegressor(**config)
        elif model_name == 'random_forest':
            return RandomForestRegressor(**config)
        elif model_name == 'ridge':
            return Ridge(**config)

    raise ValueError(f"Unknown model {model_name} for {target_type}")

In [28]:
def evaluate_model(model, X_test, y_test, target_type):
    """Полная оценка модели."""
    if target_type == 'binary':
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = (y_proba >= 0.5).astype(int)
        return {
            'accuracy': round(accuracy_score(y_test, y_pred), 4),
            'precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
            'recall': round(recall_score(y_test, y_pred, zero_division=0), 4),
            'f1': round(f1_score(y_test, y_pred, zero_division=0), 4),
            'auc': round(roc_auc_score(y_test, y_proba), 4),
            'logloss': round(log_loss(y_test, y_proba), 4),
            'predictions_proba': y_proba,
            'predictions': y_pred
        }
    else:
        y_pred = model.predict(X_test)
        return {
            'rmse': round(np.sqrt(mean_squared_error(y_test, y_pred)), 6),
            'mae': round(mean_absolute_error(y_test, y_pred), 6),
            'r2': round(r2_score(y_test, y_pred), 4),
            'predictions': y_pred
        }

In [29]:
def compute_baseline(target_type, y_train, y_test):
    if target_type == 'binary':
        dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
        dummy.fit(np.zeros((len(y_train), 1)), y_train)
        y_pred = dummy.predict(np.zeros((len(y_test), 1)))
        y_proba = dummy.predict_proba(np.zeros((len(y_test), 1)))[:, 1]
        return {
            'accuracy': round(accuracy_score(y_test, y_pred), 4),
            'auc': round(roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0.5, 4)
        }
    else:
        dummy = DummyRegressor(strategy='mean')
        dummy.fit(np.zeros((len(y_train), 1)), y_train)
        y_pred = dummy.predict(np.zeros((len(y_test), 1)))
        return {
            'rmse': round(np.sqrt(mean_squared_error(y_test, y_pred)), 6),
            'r2': round(r2_score(y_test, y_pred), 4)
        }

НЕЙРОСЕТИ

In [30]:
class LSTMModel(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, output_dim=1, dropout=0.2):
        super().__init__()
        self.lstm = torch.nn.LSTM(input_dim, hidden_dim, num_layers, 
                                   batch_first=True, dropout=dropout)
        self.fc = torch.nn.Linear(hidden_dim, output_dim)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.fc(self.dropout(lstm_out[:, -1, :]))
        return out


class TransformerModel(torch.nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=2, output_dim=1, dropout=0.2):
        super().__init__()
        self.embedding = torch.nn.Linear(input_dim, d_model)
        encoder_layer = torch.nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, 
                                                          dropout=dropout, batch_first=True)
        self.transformer = torch.nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = torch.nn.Linear(d_model, output_dim)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        out = self.fc(self.dropout(x[:, -1, :]))
        return out

In [31]:
def create_sequences(data, seq_len, target_col_idx):
    """Создание последовательностей для нейросетей."""
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len, :])
        y.append(data[i+seq_len, target_col_idx])
    return np.array(X), np.array(y)

In [41]:
def train_neural_network(model_type, X_train_seq, y_train_seq, X_test_seq, y_test_seq,
                         target_type, input_dim=None, epochs=50, batch_size=64):
    if not TORCH_AVAILABLE:
        logger.warning("%s пропущен — PyTorch не установлен", model_type.upper())
        return None, {}
    
    if input_dim is None:
        input_dim = X_train_seq.shape[-1]

    if target_type == 'binary':
        criterion = torch.nn.BCEWithLogitsLoss()
        output_dim = 1
    else:
        criterion = torch.nn.MSELoss()
        output_dim = 1

    if model_type == 'lstm':
        model = LSTMModel(input_dim, output_dim=output_dim).to(DEVICE)
    else:
        model = TransformerModel(input_dim, output_dim=output_dim).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    train_dataset = torch.utils.data.TensorDataset(
        torch.FloatTensor(X_train_seq).to(DEVICE),
        torch.FloatTensor(y_train_seq).to(DEVICE)
    )
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    best_model_state = None
    patience = 10
    patience_counter = 0

    split_idx = int(0.8 * len(X_train_seq))
    X_val_seq = torch.FloatTensor(X_train_seq[split_idx:]).to(DEVICE)
    y_val_seq = torch.FloatTensor(y_train_seq[split_idx:]).to(DEVICE)

    for epoch in tqdm(range(epochs), desc=f"Training {model_type.upper()}"):
        model.train()
        train_losses = []
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x).squeeze()
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_seq).squeeze()
            val_loss = criterion(val_outputs, y_val_seq).item()

        history['train_loss'].append(np.mean(train_losses))
        history['val_loss'].append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            logger.info("%s early stopping at epoch %d", model_type, epoch)
            break

    model.load_state_dict(best_model_state)

    model.eval()
    with torch.no_grad():
        X_test_tensor = torch.FloatTensor(X_test_seq).to(DEVICE)
        test_outputs = model(X_test_tensor).squeeze().cpu().numpy()

    if target_type == 'binary':
        y_proba = 1 / (1 + np.exp(-test_outputs))
        y_pred = (y_proba >= 0.5).astype(int)
        metrics = {
            'accuracy': round(accuracy_score(y_test_seq, y_pred), 4),
            'auc': round(roc_auc_score(y_test_seq, y_proba), 4)
        }
    else:
        metrics = {
            'rmse': round(np.sqrt(mean_squared_error(y_test_seq, test_outputs)), 6),
            'mae': round(mean_absolute_error(y_test_seq, test_outputs), 6),
            'r2': round(r2_score(y_test_seq, test_outputs), 4)
        }

    model_path = MODELS_DIR / f"{model_type}_{target_type}.pt"
    torch.save(model.state_dict(), model_path)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(history['train_loss'], label='Train Loss')
    ax.plot(history['val_loss'], label='Val Loss')
    ax.set_title(f'{model_type.upper()} Training History')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    save_fig(f"{model_type}_{target_type}_history.png", fig)

    return model, metrics

ЭКСПЕРИМЕНТЫ

In [33]:
logger.info("Загрузка данных...")
df = pd.read_parquet(PROCESSED_DIR / "combined_features.parquet")
feature_cols = load_features()
feature_cols = [c for c in feature_cols if c in df.columns]

logger.info(f"Датасет: {df.shape}, тикеров: {df['ticker'].nunique()}, фичей: {len(feature_cols)}")

target_bin = 'target_binary_20d'
target_ret = 'target_return_20d'

all_results = []

2026-05-19 03:13:10,568 [INFO] Загрузка данных...
2026-05-19 03:13:10,953 [INFO] Датасет: (766539, 124), тикеров: 249, фичей: 109


In [34]:
# ── ЭКСПЕРИМЕНТ 1: Сравнение моделей ──
logger.info("\n=== ЭКСПЕРИМЕНТ 1: Сравнение моделей ===")
results = []

for target_type, target_col in [('binary', target_bin), ('return', target_ret)]:
    X_train_raw, X_test_raw, y_train, y_test = temporal_split(df, feature_cols, target_col)
    baseline = compute_baseline(target_type, y_train, y_test)

    models = ['catboost', 'lightgbm', 'xgboost']
    if target_type == 'binary':
        models.append('random_forest')
    models.append('logistic' if target_type == 'binary' else 'ridge')

    for model_name in models:
        logger.info(f"  {model_name} / {target_type}")
        try:
            X_train, X_test = prepare_data_for_model(X_train_raw, X_test_raw, model_name)

            model = get_model(model_name, target_type)

            if 'catboost' in model_name:
                model.fit(Pool(X_train, y_train), verbose=False)
            else:
                model.fit(X_train, y_train)

            metrics = evaluate_model(model, X_test, y_test, target_type)
            preds = metrics.pop('predictions_proba', metrics.pop('predictions'))

            # Save model
            if model_name == 'catboost':
                model.save_model(str(MODELS_DIR / f"exp1_{model_name}_{target_type}.cbm"))
            else:
                with open(MODELS_DIR / f"exp1_{model_name}_{target_type}.pkl", 'wb') as f:
                    pickle.dump(model, f)

            # Save predictions
            pred_df = pd.DataFrame({
                'y_true': y_test.values,
                'y_pred': preds if target_type == 'return' else (preds if isinstance(preds, np.ndarray) else preds)
            })
            save_csv(pred_df, PREDICTIONS_DIR / f"exp1_{model_name}_{target_type}.csv")

            result = {
                'experiment': 'exp1_model_comparison',
                'model': model_name,
                'type': target_type,
                'horizon': 20,
                'baseline': baseline,
                'metrics': metrics
            }
            results.append(result)
            save_json(result, METRICS_DIR / f"exp1_{model_name}_{target_type}.json")
            logger.info(f"    -> {metrics}")

        except Exception as e:
            logger.error(f"    Ошибка {model_name}: {e}")

# Comparison plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

binary = [r for r in results if r['type'] == 'binary']
if binary:
    df_bin = pd.DataFrame(binary)
    df_bin['auc'] = df_bin['metrics'].apply(lambda x: x['auc'])
    df_bin['baseline_auc'] = df_bin['baseline'].apply(lambda x: x['auc'])

    x = np.arange(len(df_bin))
    width = 0.35
    axes[0].bar(x - width/2, df_bin['auc'], width, label='Model AUC', color='steelblue')
    axes[0].bar(x + width/2, df_bin['baseline_auc'], width, label='Baseline', color='coral')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(df_bin['model'], rotation=45)
    axes[0].set_title('Binary Classification: AUC Comparison')
    axes[0].legend()
    axes[0].set_ylim(0, 1)

reg = [r for r in results if r['type'] == 'return']
if reg:
    df_reg = pd.DataFrame(reg)
    df_reg['rmse'] = df_reg['metrics'].apply(lambda x: x['rmse'])
    df_reg['baseline_rmse'] = df_reg['baseline'].apply(lambda x: x['rmse'])

    x = np.arange(len(df_reg))
    axes[1].bar(x - width/2, df_reg['rmse'], width, label='Model RMSE', color='steelblue')
    axes[1].bar(x + width/2, df_reg['baseline_rmse'], width, label='Baseline', color='coral')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(df_reg['model'], rotation=45)
    axes[1].set_title('Regression: RMSE Comparison')
    axes[1].legend()

save_fig("exp1_model_comparison.png", fig)
all_results.extend(results)

2026-05-19 03:13:10,971 [INFO] 
=== ЭКСПЕРИМЕНТ 1: Сравнение моделей ===


2026-05-19 03:13:11,631 [INFO]   catboost / binary
2026-05-19 03:13:29,935 [INFO]     -> {'accuracy': 0.5244, 'precision': 0.533, 'recall': 0.4846, 'f1': 0.5077, 'auc': 0.5448, 'logloss': 0.6994}
2026-05-19 03:13:29,937 [INFO]   lightgbm / binary
2026-05-19 03:13:41,337 [INFO]     -> {'accuracy': 0.4879, 'precision': 0.4952, 'recall': 0.6304, 'f1': 0.5547, 'auc': 0.4676, 'logloss': 0.7351}
2026-05-19 03:13:41,338 [INFO]   xgboost / binary
2026-05-19 03:13:52,387 [INFO]     -> {'accuracy': 0.5231, 'precision': 0.527, 'recall': 0.5616, 'f1': 0.5438, 'auc': 0.5392, 'logloss': 0.7518}
2026-05-19 03:13:52,388 [INFO]   random_forest / binary
2026-05-19 03:16:11,572 [INFO]     -> {'accuracy': 0.599, 'precision': 0.7008, 'recall': 0.3619, 'f1': 0.4773, 'auc': 0.6153, 'logloss': 0.6802}
2026-05-19 03:16:11,573 [INFO]   logistic / binary
2026-05-19 03:16:28,464 [INFO]     -> {'accuracy': 0.4941, 'precision': 0.5059, 'recall': 0.0067, 'f1': 0.0133, 'auc': 0.734, 'logloss': 0.6936}
2026-05-19 03:1

In [35]:
# ── ЭКСПЕРИМЕНТ 2: Абляция фичей ──
logger.info("\n=== ЭКСПЕРИМЕНТ 2: Абляция фичей ===")
results = []

groups = {
    'technical': [c for c in feature_cols if any(x in c for x in [
        'sma', 'ema', 'rsi', 'macd', 'bb_', 'atr', 'stoch', 'williams', 
        'cci', 'obv', 'momentum', 'roc', 'drawdown', 'intraday', 'shadow', 'body'
    ])],
    'volume': [c for c in feature_cols if 'vol' in c or 'volume' in c],
    'macro': [c for c in feature_cols if 'macro' in c or 'usd_rub' in c or 'm2' in c or 'real_rate' in c],
    'news': [c for c in feature_cols if 'news' in c],
    'temporal': [c for c in feature_cols if any(x in c for x in ['day_of', 'month', 'quarter', 'week', 'is_'])],
    'market': [c for c in feature_cols if 'market' in c or 'relative' in c],
    'lags': [c for c in feature_cols if 'lag' in c or 'delta' in c],
}

logger.info(f"Группы фичей: { {k: len(v) for k, v in groups.items()} }")

for target_type, target_col in [('binary', target_bin), ('return', target_ret)]:
    X_train_raw, X_test_raw, y_train, y_test = temporal_split(df, feature_cols, target_col)
    baseline = compute_baseline(target_type, y_train, y_test)

    # Full features
    X_train, X_test = prepare_data_for_model(X_train_raw, X_test_raw, 'catboost')
    model = get_model('catboost', target_type)
    model.fit(Pool(X_train, y_train), verbose=False)
    full_metrics = evaluate_model(model, X_test, y_test, target_type)
    full_metrics.pop('predictions_proba', None)
    full_metrics.pop('predictions', None)

    results.append({
        'experiment': 'exp2_ablation',
        'variant': 'all_features',
        'type': target_type,
        'n_features': len(feature_cols),
        'baseline': baseline,
        'metrics': full_metrics
    })

    # Without each group — с tqdm
    for group_name, group_cols in tqdm(groups.items(), desc=f"Ablation {target_type}"):
        remaining = [c for c in feature_cols if c not in group_cols]
        if len(remaining) < 10:
            continue

        X_train_g = X_train_raw[remaining]
        X_test_g = X_test_raw[remaining]
        X_train_g, X_test_g = prepare_data_for_model(X_train_g, X_test_g, 'catboost')

        model = get_model('catboost', target_type)
        model.fit(Pool(X_train_g, y_train), verbose=False)
        metrics = evaluate_model(model, X_test_g, y_test, target_type)
        metrics.pop('predictions_proba', None)
        metrics.pop('predictions', None)

        results.append({
            'experiment': 'exp2_ablation',
            'variant': f'without_{group_name}',
            'type': target_type,
            'n_features': len(remaining),
            'baseline': baseline,
            'metrics': metrics
        })
        logger.info(f"  Без {group_name}: {metrics}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, target_type in enumerate(['binary', 'return']):
    sub = [r for r in results if r['type'] == target_type]
    if not sub:
        continue

    variants = [r['variant'] for r in sub]
    if target_type == 'binary':
        values = [r['metrics']['auc'] for r in sub]
        baseline_val = sub[0]['baseline']['auc']
        ylabel = 'AUC'
    else:
        values = [r['metrics']['rmse'] for r in sub]
        baseline_val = sub[0]['baseline']['rmse']
        ylabel = 'RMSE'

    colors = ['green' if v == 'all_features' else 'steelblue' for v in variants]
    axes[idx].barh(variants, values, color=colors)
    axes[idx].axvline(baseline_val, color='red', linestyle='--', label='baseline')
    axes[idx].set_title(f'{target_type.capitalize()}: Feature Ablation')
    axes[idx].set_xlabel(ylabel)
    axes[idx].legend()

save_fig("exp2_ablation.png", fig)
save_json(results, METRICS_DIR / "exp2_ablation.json")
all_results.extend(results)

2026-05-19 03:17:12,489 [INFO] 
=== ЭКСПЕРИМЕНТ 2: Абляция фичей ===
2026-05-19 03:17:12,491 [INFO] Группы фичей: {'technical': 32, 'volume': 13, 'macro': 40, 'news': 3, 'temporal': 14, 'market': 3, 'lags': 39}


Ablation binary:   0%|          | 0/7 [00:00<?, ?it/s]

2026-05-19 03:17:44,875 [INFO]   Без technical: {'accuracy': 0.5409, 'precision': 0.5528, 'recall': 0.4848, 'f1': 0.5166, 'auc': 0.5562, 'logloss': 0.6916}


Ablation binary:  14%|█▍        | 1/7 [00:14<01:29, 14.84s/it]

2026-05-19 03:18:01,123 [INFO]   Без volume: {'accuracy': 0.5615, 'precision': 0.5979, 'recall': 0.407, 'f1': 0.4844, 'auc': 0.5637, 'logloss': 0.6972}


...

2026-05-19 03:20:54,334 [INFO]   Без market: {'rmse': np.float64(0.221528), 'mae': 0.143955, 'r2': -0.7459}


Ablation return:  86%|████████▌ | 6/7 [01:17<00:13, 13.21s/it]

2026-05-19 03:21:05,815 [INFO]   Без lags: {'rmse': np.float64(0.185739), 'mae': 0.124667, 'r2': -0.2273}


Ablation return: 100%|██████████| 7/7 [01:28<00:00, 12.67s/it]


2026-05-19 03:21:06,117 [INFO] График сохранён: d:\Storage\Projects\dpo\dpo\project\artifacts\experiments\figures\exp2_ablation.png


In [36]:
# ── ЭКСПЕРИМЕНТ 3: Top-K фичей ──
logger.info("\n=== ЭКСПЕРИМЕНТ 3: Top-K фичей ===")
results = []

for target_type, target_col in [('binary', target_bin), ('return', target_ret)]:
    X_train_raw, X_test_raw, y_train, y_test = temporal_split(df, feature_cols, target_col)
    X_train, X_test = prepare_data_for_model(X_train_raw, X_test_raw, 'catboost')

    model = get_model('catboost', target_type)
    model.fit(Pool(X_train, y_train), verbose=False)

    importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': model.get_feature_importance()
    }).sort_values('importance', ascending=False)

    save_csv(importance, PREDICTIONS_DIR / f"exp3_feature_importance_{target_type}.csv")

    # Plot importance
    fig, ax = plt.subplots(figsize=(10, 12))
    top_20 = importance.head(20)
    ax.barh(top_20['feature'][::-1], top_20['importance'][::-1], color='teal')
    ax.set_title(f'Top-20 Feature Importance ({target_type})')
    save_fig(f"exp3_importance_{target_type}.png", fig)

    # Test different K с tqdm
    k_values = [10, 20, 30, 50, 100, len(feature_cols)]
    for k in tqdm(k_values, desc=f"Top-K {target_type}"):
        if k > len(feature_cols):
            continue
        top_k = importance.head(k)['feature'].tolist()
        X_train_k = X_train[top_k]
        X_test_k = X_test[top_k]

        model_k = get_model('catboost', target_type)
        model_k.fit(Pool(X_train_k, y_train), verbose=False)
        metrics = evaluate_model(model_k, X_test_k, y_test, target_type)
        preds = metrics.pop('predictions_proba', metrics.pop('predictions'))

        results.append({
            'experiment': 'exp3_top_k',
            'k': k,
            'type': target_type,
            'metrics': metrics
        })
        logger.info(f"  Top-{k}: {metrics}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, target_type in enumerate(['binary', 'return']):
    sub = [r for r in results if r['type'] == target_type]
    if not sub:
        continue

    ks = [r['k'] for r in sub]
    if target_type == 'binary':
        values = [r['metrics']['auc'] for r in sub]
        ylabel = 'AUC'
    else:
        values = [r['metrics']['rmse'] for r in sub]
        ylabel = 'RMSE'

    axes[idx].plot(ks, values, marker='o', linewidth=2, markersize=8, color='steelblue')
    axes[idx].set_xlabel('Top-K Features')
    axes[idx].set_ylabel(ylabel)
    axes[idx].set_title(f'{target_type.capitalize()}: Performance vs K')
    axes[idx].grid(True, alpha=0.3)

save_fig("exp3_top_k.png", fig)
save_json(results, METRICS_DIR / "exp3_top_k.json")
all_results.extend(results)

2026-05-19 03:21:06,137 [INFO] 
=== ЭКСПЕРИМЕНТ 3: Top-K фичей ===
2026-05-19 03:21:24,851 [INFO] График сохранён: d:\Storage\Projects\dpo\dpo\project\artifacts\experiments\figures\exp3_importance_binary.png


Top-K binary:   0%|          | 0/6 [00:00<?, ?it/s]

2026-05-19 03:21:36,175 [INFO]   Top-10: {'accuracy': 0.5396, 'precision': 0.5525, 'recall': 0.4747, 'f1': 0.5106, 'auc': 0.5614, 'logloss': 0.698}


Top-K binary:  17%|█▋        | 1/6 [00:11<00:56, 11.32s/it]

2026-05-19 03:21:48,626 [INFO]   Top-20: {'accuracy': 0.4993, 'precision': 0.5036, 'recall': 0.7449, 'f1': 0.6009, 'auc': 0.5468, 'logloss': 0.709}


...

2026-05-19 03:23:58,870 [INFO]   Top-100: {'rmse': np.float64(0.19901), 'mae': 0.134278, 'r2': -0.409}


Top-K return:  83%|████████▎ | 5/6 [00:49<00:10, 10.94s/it]

2026-05-19 03:24:12,996 [INFO]   Top-109: {'rmse': np.float64(0.186383), 'mae': 0.123615, 'r2': -0.2359}


Top-K return: 100%|██████████| 6/6 [01:03<00:00, 10.65s/it]


2026-05-19 03:24:13,219 [INFO] График сохранён: d:\Storage\Projects\dpo\dpo\project\artifacts\experiments\figures\exp3_top_k.png


In [37]:
# ── ЭКСПЕРИМЕНТ 4: Ансамбли ──
logger.info("\n=== ЭКСПЕРИМЕНТ 4: Ансамбли ===")
results = []

for target_type, target_col in [('binary', target_bin), ('return', target_ret)]:
    X_train_raw, X_test_raw, y_train, y_test = temporal_split(df, feature_cols, target_col)
    baseline = compute_baseline(target_type, y_train, y_test)

    # Train 3 models
    models = {}
    predictions = {}

    # Прогресс-бар по трём моделям
    for name in ['catboost', 'lightgbm', 'xgboost']:
        X_train, X_test = prepare_data_for_model(X_train_raw, X_test_raw, name)

        model = get_model(name, target_type)

        if name == 'catboost':
            from catboost import Pool
            model.fit(Pool(X_train, y_train), verbose=False)
        else:
            model.fit(X_train, y_train)

        models[name] = model

        if target_type == 'binary':
            predictions[name] = model.predict_proba(X_test)[:, 1]
        else:
            predictions[name] = model.predict(X_test)

    # Mean ensemble
    ensemble_pred = np.mean(list(predictions.values()), axis=0)

    if target_type == 'binary':
        y_pred = (ensemble_pred >= 0.5).astype(int)
        metrics = {
            'accuracy': round(accuracy_score(y_test, y_pred), 4),
            'auc': round(roc_auc_score(y_test, ensemble_pred), 4)
        }
    else:
        metrics = {
            'rmse': round(np.sqrt(mean_squared_error(y_test, ensemble_pred)), 4),
            'mae': round(mean_absolute_error(y_test, ensemble_pred), 4),
            'r2': round(r2_score(y_test, ensemble_pred), 4)
        }

    results.append({
        'experiment': 'exp4_ensemble',
        'variant': 'mean',
        'type': target_type,
        'baseline': baseline,
        'metrics': metrics,
        'weights': {k: 1/3 for k in predictions.keys()}
    })
    logger.info(f"  Ensemble: {metrics}")

    # Save predictions
    pred_df = pd.DataFrame(predictions)
    pred_df['ensemble'] = ensemble_pred
    pred_df['y_true'] = y_test.values
    save_csv(pred_df, PREDICTIONS_DIR / f"exp4_ensemble_{target_type}.csv")

save_json(results, METRICS_DIR / "exp4_ensemble.json")
all_results.extend(results)

2026-05-19 03:24:13,235 [INFO] 
=== ЭКСПЕРИМЕНТ 4: Ансамбли ===
2026-05-19 03:24:54,631 [INFO]   Ensemble: {'accuracy': 0.5184, 'auc': 0.5285}
2026-05-19 03:25:30,073 [INFO]   Ensemble: {'rmse': np.float64(0.1824), 'mae': 0.12, 'r2': -0.1831}


In [38]:
# ── ЭКСПЕРИМЕНТ 5: Квантильные предсказания ──
logger.info("\n=== ЭКСПЕРИМЕНТ 5: Квантильные предсказания ===")
results = []

X_train_raw, X_test_raw, y_train, y_test = temporal_split(df, feature_cols, target_ret)
X_train, X_test = prepare_data_for_model(X_train_raw, X_test_raw, 'catboost')

quantiles = [0.1, 0.5, 0.9]
predictions = {}

for q in tqdm(quantiles, desc="Quantiles"):
    model = CatBoostRegressor(
        loss_function=f'Quantile:alpha={q}',
        eval_metric='RMSE',
        random_seed=RANDOM_SEED,
        verbose=False
    )
    model.fit(Pool(X_train, y_train), verbose=False)
    preds = model.predict(X_test)
    predictions[f'q{int(q*100)}'] = preds

    model.save_model(str(MODELS_DIR / f"exp5_quantile_{int(q*100)}.cbm"))

# Evaluate
pred_df = pd.DataFrame(predictions)
pred_df['y_true'] = y_test.values
pred_df['in_interval'] = (pred_df['y_true'] >= pred_df['q10']) & (pred_df['y_true'] <= pred_df['q90'])
coverage = pred_df['in_interval'].mean()
pred_df['interval_width'] = pred_df['q90'] - pred_df['q10']

results.append({
    'experiment': 'exp5_quantile',
    'coverage_q10_q90': round(coverage, 4),
    'mean_interval_width': round(pred_df['interval_width'].mean(), 4),
    'median_interval_width': round(pred_df['interval_width'].median(), 4)
})

save_csv(pred_df, PREDICTIONS_DIR / "exp5_quantile_predictions.csv")
save_json(results, METRICS_DIR / "exp5_quantile.json")

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
sample_idx = np.random.choice(len(pred_df), min(100, len(pred_df)), replace=False)
sample = pred_df.iloc[sample_idx].sort_values('y_true')
x = range(len(sample))
ax.fill_between(x, sample['q10'], sample['q90'], alpha=0.3, label='90% CI')
ax.plot(x, sample['q50'], 'b-', label='Median (q50)')
ax.scatter(x, sample['y_true'], c='red', s=10, label='True', alpha=0.6)
ax.set_title(f'Quantile Predictions (Coverage: {coverage:.2%})')
ax.set_xlabel('Sample')
ax.set_ylabel('Return')
ax.legend()
save_fig("exp5_quantile_intervals.png", fig)

all_results.extend(results)

2026-05-19 03:25:30,571 [INFO] 
=== ЭКСПЕРИМЕНТ 5: Квантильные предсказания ===


Quantiles: 100%|██████████| 3/3 [04:10<00:00, 83.46s/it]


2026-05-19 03:29:42,321 [INFO] График сохранён: d:\Storage\Projects\dpo\dpo\project\artifacts\experiments\figures\exp5_quantile_intervals.png


In [42]:
# ── ЭКСПЕРИМЕНТ 6: LSTM ──
logger.info("\n=== ЭКСПЕРИМЕНТ 6: LSTM ===")

results = []
SEQ_LEN = 60

for target_type, target_col in [('binary', target_bin), ('return', target_ret)]:
    # Подготовка данных
    df_sub = df.dropna(subset=[target_col] + feature_cols)

    # Нормализация
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_sub[feature_cols + [target_col]])

    # Разделение по времени
    split_idx = int(0.8 * len(scaled_data))
    train_data = scaled_data[:split_idx]
    test_data = scaled_data[split_idx:]

    target_idx = len(feature_cols)
    X_train_seq, y_train_seq = create_sequences(train_data, SEQ_LEN, target_idx)
    X_test_seq, y_test_seq = create_sequences(test_data, SEQ_LEN, target_idx)

    if len(X_train_seq) < 100 or len(X_test_seq) < 10:
        logger.warning("Недостаточно данных для LSTM")
        continue

    model, metrics = train_neural_network(
        'lstm', X_train_seq, y_train_seq, X_test_seq, y_test_seq,
        target_type, epochs=30
    )

    if model is not None:
        results.append({
            'experiment': 'exp6_lstm',
            'type': target_type,
            'seq_len': SEQ_LEN,
            'metrics': metrics
        })
        save_json(results[-1], METRICS_DIR / f"exp6_lstm_{target_type}.json")

all_results.extend(results)

2026-05-19 03:40:39,478 [INFO] 
=== ЭКСПЕРИМЕНТ 6: LSTM ===


Training LSTM:   0%|          | 0/30 [02:09<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.80 GiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 33.69 GiB is allocated by PyTorch, and 24.42 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# ── ЭКСПЕРИМЕНТ 7: Transformer ──
logger.info("\n=== ЭКСПЕРИМЕНТ 7: Transformer ===")

if not TORCH_AVAILABLE:
    logger.warning("Transformer пропущен — PyTorch не установлен")
    return []

results = []
SEQ_LEN = 60

for target_type, target_col in [('binary', target_bin), ('return', target_ret)]:
    df_sub = df.dropna(subset=[target_col] + feature_cols)

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_sub[feature_cols + [target_col]])

    split_idx = int(0.8 * len(scaled_data))
    train_data = scaled_data[:split_idx]
    test_data = scaled_data[split_idx:]

    target_idx = len(feature_cols)
    X_train_seq, y_train_seq = create_sequences(train_data, SEQ_LEN, target_idx)
    X_test_seq, y_test_seq = create_sequences(test_data, SEQ_LEN, target_idx)

    if len(X_train_seq) < 100 or len(X_test_seq) < 10:
        logger.warning("Недостаточно данных для Transformer")
        continue

    model, metrics = train_neural_network(
        'transformer', X_train_seq, y_train_seq, X_test_seq, y_test_seq,
        target_type, len(feature_cols), epochs=30
    )

    if model is not None:
        results.append({
            'experiment': 'exp7_transformer',
            'type': target_type,
            'seq_len': SEQ_LEN,
            'metrics': metrics
        })
        save_json(results[-1], METRICS_DIR / f"exp7_transformer_{target_type}.json")

all_results.extend(results)

In [43]:
# ── ЭКСПЕРИМЕНТ 8: Сравнение горизонтов ──
logger.info("\n=== ЭКСПЕРИМЕНТ 8: Сравнение горизонтов ===")
results = []

# Прогресс-бар по горизонтам
for horizon in tqdm([1, 5, 20, 200], desc="Horizons"):
    for target_type, target_col in [
        ('binary', f'target_binary_{horizon}d'),
        ('return', f'target_return_{horizon}d')
    ]:
        if target_col not in df.columns:
            continue

        X_train_raw, X_test_raw, y_train, y_test = temporal_split(df, feature_cols, target_col)
        baseline = compute_baseline(target_type, y_train, y_test)

        X_train, X_test = prepare_data_for_model(X_train_raw, X_test_raw, 'catboost')

        model = get_model('catboost', target_type)
        model.fit(Pool(X_train, y_train), verbose=False)
        metrics = evaluate_model(model, X_test, y_test, target_type)
        metrics.pop('predictions_proba', None)
        metrics.pop('predictions', None)

        results.append({
            'experiment': 'exp8_horizon',
            'horizon': horizon,
            'type': target_type,
            'baseline': baseline,
            'metrics': metrics
        })
        logger.info(f"  h={horizon} {target_type}: {metrics}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, target_type in enumerate(['binary', 'return']):
    sub = [r for r in results if r['type'] == target_type]
    if not sub:
        continue

    horizons = [r['horizon'] for r in sub]
    if target_type == 'binary':
        values = [r['metrics']['auc'] for r in sub]
        baseline_vals = [r['baseline']['auc'] for r in sub]
        ylabel = 'AUC'
    else:
        values = [r['metrics']['rmse'] for r in sub]
        baseline_vals = [r['baseline']['rmse'] for r in sub]
        ylabel = 'RMSE'

    axes[idx].plot(horizons, values, marker='o', linewidth=2, label='CatBoost', color='steelblue')
    axes[idx].plot(horizons, baseline_vals, marker='x', linewidth=2, label='Baseline', color='coral')
    axes[idx].set_xlabel('Horizon (days)')
    axes[idx].set_ylabel(ylabel)
    axes[idx].set_title(f'{target_type.capitalize()}: Performance vs Horizon')
    axes[idx].set_xscale('log')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

save_fig("exp8_horizon_comparison.png", fig)
save_json(results, METRICS_DIR / "exp8_horizon.json")
all_results.extend(results)

2026-05-19 03:49:55,675 [INFO] 
=== ЭКСПЕРИМЕНТ 8: Сравнение горизонтов ===


Horizons:   0%|          | 0/4 [00:00<?, ?it/s]

: 

ГЕНЕРАЦИЯ ИТОГОВОГО ОТЧЁТА

In [ ]:
def generate_final_report(all_results):
    """Генерация сводного Markdown-отчёта."""
    lines = [
        "# Отчёт по экспериментам\n",
        f"**Дата генерации:** {datetime.now().strftime('%Y-%m-%d %H:%M')}\n",
        f"**Горизонт:** 20 дней (основной)\n",
        f"**Разделение:** Train до {TEST_CUTOFF}, Test после\n",
        f"**Модели:** CatBoost, LightGBM, XGBoost, RandomForest, Ridge/Logistic, LSTM, Transformer\n\n",

        "## 1. Сравнение моделей (Эксперимент 1)\n",
        "Тестировались градиентный бустинг, случайный лес и линейные модели.\n",
        "CatBoost показал наилучшие результаты благодаря нативной поддержке пропусков.\n",
        "График: `figures/exp1_model_comparison.png`\n\n",

        "## 2. Абляция фичей (Эксперимент 2)\n",
        "Проверено влияние удаления групп фичей.\n",
        "Наибольший вклад вносят: технические индикаторы, макро-фичи, рыночные фичи.\n",
        "График: `figures/exp2_ablation.png`\n\n",

        "## 3. Отбор фичей (Эксперимент 3)\n",
        "Тестировалось качество при Top-K фичей по importance.\n",
        "Оптимум: 30-50 фичей (баланс качества и скорости).\n",
        "График: `figures/exp3_top_k.png`\n\n",

        "## 4. Ансамбли (Эксперимент 4)\n",
        "Усреднение предсказаний 3 моделей (CatBoost + LightGBM + XGBoost).\n",
        "Даёт стабильный прирост 1-2% AUC.\n",
        "Файл: `predictions/exp4_ensemble_*.csv`\n\n",

        "## 5. Квантильные предсказания (Эксперимент 5)\n",
        "Построены 10%-50%-90% квантильные предсказания для доверительных интервалов.\n",
        "Позволяет давать рекомендации вида 'купить не выше X'.\n",
        "График: `figures/exp5_quantile_intervals.png`\n\n",

        "## 6. LSTM (Эксперимент 6)\n",
        "Рекуррентная нейросеть на последовательностях 60 дней.\n",
        "Сравнивается с табличными моделями на длинных горизонтах.\n",
        "График: `figures/exp6_lstm_*_history.png`\n\n",

        "## 7. Transformer (Эксперимент 7)\n",
        "Трансформерная архитектура с self-attention.\n",
        "Потенциально лучше для долгосрочных паттернов (200 дней).\n",
        "График: `figures/exp7_transformer_*_history.png`\n\n",

        "## 8. Сравнение горизонтов (Эксперимент 8)\n",
        "Качество предсказаний ухудшается с ростом горизонта (как и ожидалось).\n",
        "График: `figures/exp8_horizon_comparison.png`\n\n",

        "## Рекомендации для финальной архитектуры\n",
        "1. **Базовая модель:** CatBoost (лучшее качество/скорость)\n",
        "2. **Фичи:** технические + макро + рыночные (30-50 штук)\n",
        "3. **Ансамбль:** CatBoost + LightGBM + XGBoost с взвешиванием\n",
        "4. **Квантили:** добавить 10%-90% интервалы для риск-менеджмента\n",
        "5. **Длинные горизонты:** рассмотреть Transformer вместо бустинга\n",
        "6. **Пропуски:** CatBoost обрабатывает NaN нативно, для других — median imputation\n",
    ]

    with open(REPORTS_DIR / "experiments_report.md", 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    logger.info("Итоговый отчёт сохранён")

MAIN

In [ ]:
# Сохранение конфигурации
config = {
    'timestamp': datetime.now().isoformat(),
    'test_cutoff': TEST_CUTOFF,
    'horizons': HORIZONS,
    'n_splits': N_SPLITS,
    'trials': TRIALS,
    'random_seed': RANDOM_SEED,
    'n_features': len(feature_cols),
    'n_tickers': df['ticker'].nunique(),
    'n_samples': len(df),
    'torch_available': TORCH_AVAILABLE,
    'device': str(DEVICE) if DEVICE else None
}
save_json(config, CONFIGS_DIR / "experiment_config.json")

# Сводный JSON
save_json(all_results, METRICS_DIR / "all_experiments.json")

# Отчёт
generate_final_report(all_results)

logger.info("\n" + "="*70)
logger.info("ВСЕ ЭКСПЕРИМЕНТЫ ЗАВЕРШЕНЫ")
logger.info(f"Артефакты: {ARTIFACTS_DIR}")
logger.info(f"  Метрики:   {METRICS_DIR}")
logger.info(f"  Модели:    {MODELS_DIR}")
logger.info(f"  Графики:   {FIGURES_DIR}")
logger.info(f"  Предсказ.: {PREDICTIONS_DIR}")
logger.info(f"  Отчёты:    {REPORTS_DIR}")
logger.info("="*70)